# NaviKurd: 80 everyday object categories
Run top to bottom in a GPU runtime with persistent storage with internet access. Source: [COCO 2017](https://cocodataset.org/#download). Download official annotations and only the selected training photographs. Budget tens of GB of disk space for 100,000 distinct photos and persist the dataset and training runs between sessions. This is a course experiment, not a navigation aid.

In [ ]:
%pip -q install ultralytics==8.3.228

In [ ]:
import json, random, zipfile, shutil, yaml, csv
from pathlib import Path
from urllib.request import urlretrieve
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict, Counter
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO
root = Path('navikurd_data/coco100k_all80_v1').resolve(); root.mkdir(exist_ok=True)
z = root/'annotations.zip'
if not z.exists(): urlretrieve('https://images.cocodataset.org/annotations/annotations_trainval2017.zip', z)
with zipfile.ZipFile(z) as f: data = json.load(f.open('annotations/instances_train2017.json'))
categories = sorted(data['categories'], key=lambda x: x['id'])
names = [x['name'] for x in categories]
classes = {x['id']: k for k, x in enumerate(categories)}
assert len(classes) == len(names) == 80
print(', '.join(names))
boxes = defaultdict(list); crowd = set()
for a in data['annotations']:
    if a['category_id'] in classes:
        if a['iscrowd']: crowd.add(a['image_id'])
        else: boxes[a['image_id']].append((classes[a['category_id']], a['bbox']))
images = {x['id']:x for x in data['images']}; del data

Sample 100,000 unique photos without replacement from images without crowd annotations. Keep images with no annotated objects eligible as background. Split by image ID, not by box.

In [ ]:
rng = random.Random(42)
eligible = sorted(images.keys() - crowd)
total = 100_000
assert len(eligible) >= total
ids = rng.sample(eligible, total)
splits = dict(train=ids[:70000], val=ids[70000:85000], test=ids[85000:])
assert len(set(ids)) == len(ids) == total
for split, subset in splits.items():
    counts = Counter(names[k] for i in subset for k,_ in boxes.get(i, []))
    print(split, len(subset), 'images;', sum(not boxes.get(i) for i in subset), 'background images;', counts)
    assert len(counts) == len(names)
(root/'split_ids.json').write_text(json.dumps(splits))

In [ ]:
def prepare(job):
    split,i=job; info=images[i]; w,h=info['width'],info['height']; filename=info['file_name']
    img=root/'images'/split/filename; label=root/'labels'/split/(img.stem+'.txt')
    if not img.exists():
        tmp=img.with_suffix('.part'); urlretrieve('https://images.cocodataset.org/train2017/'+filename,tmp); tmp.replace(img)
    rows=[]
    for k,(x,y,bw,bh) in boxes[i]:
        rows.append(f'{k} {(x+bw/2)/w} {(y+bh/2)/h} {bw/w} {bh/h}')
    label.write_text('\n'.join(rows))
for s in splits:
    for d in ['images','labels']: (root/d/s).mkdir(parents=True,exist_ok=True)
with ThreadPoolExecutor(6) as pool: list(pool.map(prepare,[(s,i) for s in splits for i in splits[s]]))
cfg=root/'coco80.yaml'; cfg.write_text(yaml.safe_dump(dict(path=str(root),names=names,**{s:'images/'+s for s in splits})))

In [ ]:
from matplotlib.patches import Rectangle
fig,axs=plt.subplots(2,5,figsize=(16,7))
for ax,i in zip(axs.flat,[i for i in splits['train'] if boxes.get(i)][:10]):
    ax.imshow(Image.open(root/'images/train'/images[i]['file_name'])); ax.axis('off')
    for k,(x,y,w,h) in boxes[i]:
        ax.add_patch(Rectangle((x,y),w,h,fill=False,color='red')); ax.text(x,y,names[k],backgroundcolor='white')
plt.tight_layout()

Train a small YOLO detector from random initialization to avoid COCO pretraining overlap. Validation selects the best epoch; inspect loss curves for overfitting. Inspect per-class results because common classes can hide poor results on rare ones.

In [ ]:
model=YOLO('yolo11n.yaml'); model.train(data=str(cfg),epochs=40,patience=8,imgsz=416,batch=8,seed=42,workers=2,pretrained=False)
run=Path(model.trainer.save_dir); display(Image.open(run/'results.png'))
best=YOLO(str(run/'weights/best.pt'))

Final held-out evaluation: run after freezing all settings. AP measures detection quality across confidence thresholds. Discuss weak classes and example errors in your report; do not tune on these results.

In [ ]:
scores=best.val(data=str(cfg),split='test',imgsz=416,plots=True); print('Test mAP50-95:',scores.box.map)
for k,ap in zip(scores.box.ap_class_index,scores.box.ap): print(names[k],float(ap))
i=splits['test'][0]; prediction=best(str(root/'images/test'/images[i]['file_name']))[0]; plt.imshow(prediction.plot()[:,:,::-1]); plt.axis('off');